# Evaluación de generación de texto con **BLEU, ROUGE y METEOR**

**Trabajo final MIA-10 · Procesamiento del Lenguaje Natural — Dr. Wester Zela Moraya**
Autor: Carlos Pérez Pérez · Maestría en IA, UNI

## Por qué estas métricas necesitan una tarea *generativa*

BLEU, ROUGE y METEOR miden **cuánto se parece un texto generado por el modelo a un texto de referencia**
(se usan en traducción y resumen). **No** aplican a un clasificador (nuestro modelo de *naturaleza* del
evento se evalúa con accuracy/F1/kappa). Por eso, para usarlas correctamente, planteamos una **tarea de
resumen** dentro del mismo proyecto:

> **Tarea:** resumir automáticamente la epicrisis en un **resumen del curso clínico**.
> **Referencia (gold):** la sección **"Brief Hospital Course"** de la propia nota (un resumen escrito por
> el médico → referencia legítima).
> **Candidatos:** (A) un baseline extractivo *Lead-3* y (B) un modelo abstractive **BART**.
> **Métricas:** BLEU · ROUGE-1/2/L · METEOR.

Comparar un baseline contra un modelo real es lo que da sentido a las cifras (un número solo no dice nada).


## 1. Configuración

In [ ]:
import os, re, sys
from pathlib import Path

BASE = Path.home() / "tesis-gemses-mimic-pipeline"
DATASET_PARQUET = BASE / "04_pipeline_codigo" / "datos_intermedios" / "fase4" / "fase4_dataset.parquet"

N_SAMPLE = 30                                  # notas a evaluar (sube si tienes GPU y tiempo)
MODEL    = "sshleifer/distilbart-cnn-12-6"     # resumidor ligero; alterna: facebook/bart-large-cnn
SHOW_TEXT = True                               # ponlo en False antes de exportar para el profesor (DUA)

try:
    import torch
    GPU = torch.cuda.is_available()
    print("GPU:", GPU, "-", (torch.cuda.get_device_name(0) if GPU else "solo CPU"))
except Exception as e:
    GPU = False
    print("torch aun no disponible:", e)


## 2. Dependencias

In [ ]:
import importlib, subprocess
def ensure(mod, pip_name=None):
    try:
        importlib.import_module(mod)
    except ImportError:
        print("instalando", pip_name or mod, "...")
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pip_name or mod])
for mod, name in [("pandas","pandas"), ("transformers","transformers"), ("torch","torch"),
                  ("sacrebleu","sacrebleu"), ("rouge_score","rouge-score"), ("nltk","nltk")]:
    ensure(mod, name)

import nltk
for pkg in ["wordnet", "omw-1.4", "punkt", "punkt_tab"]:
    try:
        nltk.download(pkg, quiet=True)
    except Exception:
        pass
print("Dependencias listas.")


## 3. Datos: fuente y referencia

Si está el parquet real, se extrae la sección **"Brief Hospital Course"** como referencia y se resume el
resto de la nota. Si no, corre con un **mini-ejemplo sintético** (sin datos de pacientes) para que veas las
métricas funcionando igual.

In [ ]:
import pandas as pd

def extraer_seccion(nota, header="Brief Hospital Course"):
    m = re.search(re.escape(header) + r"\s*:?\s*(.+?)(?=\n\s*[A-Z][A-Za-z /]{2,40}:|\Z)", nota, re.S)
    return m.group(1).strip() if m else None

USE_REAL = DATASET_PARQUET.exists()
if USE_REAL:
    df = pd.read_parquet(DATASET_PARQUET)                 # note_id, text, naturaleza
    df["reference"] = df["text"].apply(extraer_seccion)
    df = df[df["reference"].fillna("").str.len() >= 120].head(N_SAMPLE).reset_index(drop=True)
    # fuente = nota sin la seccion de referencia (evita copiar la respuesta)
    df["source"] = df.apply(lambda r: r["text"].replace(r["reference"], " "), axis=1)
    sources    = df["source"].tolist()
    references = df["reference"].tolist()
    print(f"Datos REALES: {len(sources)} notas con 'Brief Hospital Course'.")
else:
    ejemplos = [
        ("The patient was admitted with fever and dysuria. Urine culture grew E. coli. "
         "IV ceftriaxone was started and the patient defervesced by day 2. Renal function "
         "remained stable. He was transitioned to oral antibiotics and discharged.",
         "Patient admitted for a urinary tract infection with E. coli, treated with "
         "ceftriaxone then oral antibiotics, and discharged in stable condition."),
        ("An 68-year-old woman presented with chest pain. Troponin was elevated and ECG "
         "showed ST changes. She underwent cardiac catheterization with a drug-eluting "
         "stent to the LAD. Dual antiplatelet therapy was initiated.",
         "Woman with NSTEMI treated with a drug-eluting stent to the LAD and started on "
         "dual antiplatelet therapy."),
        ("The patient developed hypoglycemia after an insulin dose. Blood glucose was 42. "
         "Dextrose was administered with rapid recovery. Insulin regimen was adjusted and "
         "no further episodes occurred.",
         "An adverse medication event of hypoglycemia after insulin, corrected with dextrose "
         "and an adjusted insulin regimen."),
        ("Postoperatively the patient had a surgical site infection. Wound cultures grew MRSA. "
         "Vancomycin was started and the wound was debrided. The patient improved and was "
         "discharged on oral antibiotics.",
         "Postoperative MRSA surgical site infection managed with vancomycin and debridement, "
         "discharged on oral antibiotics."),
    ]
    sources    = [e[0] for e in ejemplos]
    references = [e[1] for e in ejemplos]
    print(f"Datos SINTETICOS de ejemplo: {len(sources)} pares (sin datos de pacientes).")
print("Referencia de ejemplo:", references[0][:160], "...")


## 4. Dos sistemas de resumen
(A) *Lead-3* extractivo (baseline) · (B) BART abstractive (modelo)

In [ ]:
# (A) Baseline extractivo: las primeras 3 oraciones
def lead_n(texto, n=3):
    sents = re.split(r"(?<=[.!?])\s+", texto.strip())
    return " ".join(sents[:n]).strip()

cands_lead = [lead_n(s) for s in sources]
print("Lead-3 listo. Ejemplo:", cands_lead[0][:160], "...")


In [ ]:
# (B) Modelo abstractive: BART
from transformers import pipeline
device = 0 if GPU else -1
summarizer = pipeline("summarization", model=MODEL, device=device)

def resumir(texto):
    out = summarizer(texto[:3000], max_length=120, min_length=25, truncation=True)
    return out[0]["summary_text"].strip()

cands_bart = [resumir(s) for s in sources]
print("BART listo. Ejemplo:", cands_bart[0][:160], "...")


## 5. Cálculo de BLEU, ROUGE y METEOR

In [ ]:
import sacrebleu
from rouge_score import rouge_scorer
from nltk.translate.meteor_score import meteor_score
import nltk

def tok(s):
    try:
        return nltk.word_tokenize(s)
    except Exception:
        return s.split()

_rs = rouge_scorer.RougeScorer(["rouge1", "rouge2", "rougeL"], use_stemmer=True)

def evaluar(cands, refs, nombre):
    bleu = sacrebleu.corpus_bleu(cands, [refs]).score           # 0-100
    r1 = r2 = rl = met = 0.0
    for c, r in zip(cands, refs):
        sc = _rs.score(r, c)                                    # (referencia, candidato)
        r1 += sc["rouge1"].fmeasure
        r2 += sc["rouge2"].fmeasure
        rl += sc["rougeL"].fmeasure
        met += meteor_score([tok(r)], tok(c))
    n = len(cands)
    return {"sistema": nombre, "BLEU": round(bleu, 2),
            "ROUGE-1": round(100 * r1 / n, 2), "ROUGE-2": round(100 * r2 / n, 2),
            "ROUGE-L": round(100 * rl / n, 2), "METEOR": round(100 * met / n, 2)}

filas = [evaluar(cands_lead, references, "Lead-3 (baseline)"),
         evaluar(cands_bart, references, f"BART ({MODEL.split('/')[-1]})")]
tabla = pd.DataFrame(filas)[["sistema", "BLEU", "ROUGE-1", "ROUGE-2", "ROUGE-L", "METEOR"]]
tabla


## 6. Ejemplos cualitativos
(Con datos reales, pon `SHOW_TEXT = False` en la celda 1 antes de exportar para el profesor.)

In [ ]:
if SHOW_TEXT:
    for i in range(min(2, len(sources))):
        print("="*90)
        print("REFERENCIA (Brief Hospital Course):\n ", references[i][:300])
        print("\nLead-3:\n ", cands_lead[i][:300])
        print("\nBART:\n ", cands_bart[i][:300])
else:
    print("SHOW_TEXT=False: se omiten los textos (modo seguro para compartir con el profesor).")


## 7. Cómo leer los resultados (y qué NO afirman)

- **BLEU** premia la coincidencia de n-gramas (precisión); es duro con el resumen abstractive porque
  reformula. **ROUGE** mide solape (recall/F) — es la métrica estándar de *resumen*; **ROUGE-L** captura la
  subsecuencia común más larga. **METEOR** añade sinónimos y raíces, por eso suele ser más alto.
- Se reportan en escala 0–100 (ROUGE y METEOR ×100) para compararlas de un vistazo.
- **Limitaciones honestas:** (1) la referencia es la sección *Brief Hospital Course* como **proxy** de un
  resumen ideal; (2) estas métricas miden **solape léxico**, no corrección clínica — un resumen puede
  puntuar alto y estar clínicamente incompleto; (3) muestra pequeña (`N_SAMPLE`), sin intervalos.

**Para el profesor:** exporta con `SHOW_TEXT = False` para que el HTML muestre solo la tabla de métricas,
sin texto de pacientes:
```bash
jupyter nbconvert --to html Eval_BLEU_ROUGE_METEOR.ipynb
```

*Curso MIA-10 · Maestría en IA · UNI · 2026.*
